# ML-Filtered Intraday Mean Reversion Research Demo

This notebook mirrors the production research path used by the Streamlit app. The goal is not to tune a backtest until it looks good; the goal is to evaluate a clearly defined mean-reversion hypothesis with realistic execution timing, explicit portfolio constraints, and walk-forward out-of-sample validation.

Research question: after a large short-term selloff in liquid US equities, is there a measurable next-session intraday reversal premium, and does an interpretable ML filter improve trade selection?

## 1. Research Setup

The raw signal defines the economic hypothesis. Logistic Regression is used only as a calibrated filter on candidate trades.

Key assumptions:

- Signal is observed at the close of day `t`.
- Entry is at `Open[t+1]`.
- Exit is at `Close[t+1]`.
- Transaction cost is charged as a fixed round-trip cost.
- Walk-forward folds train only on history available before each test window.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import data, features, model, metrics, plots, walkforward

## 2. Parameters

Only the mean-reversion threshold should be viewed as the core hypothesis parameter. The ML threshold, cost model, and walk-forward cadence are fixed to reduce overfitting risk.

In [ ]:
tickers = ['AAPL', 'MSFT', 'NVDA', 'TSLA', 'AMZN']
benchmark_ticker = 'SPY'

start_date = '2018-01-01'
end_date = '2025-12-31'
use_cache = True
force_refresh = False

initial_capital = 100_000
return_threshold = -0.05
ml_prob_threshold = 0.50
max_positions = 3
max_weight = 0.50
transaction_cost = 0.001
risk_free_rate = 0.04
min_train_years = 3
test_window_months = 12

## 3. Load Data

The data loader checks local CSV cache first. If a requested range is not covered, it downloads through Yahoo Finance and updates the cache.

In [ ]:
df_raw = data.download_data(
    tickers,
    start_date,
    end_date,
    use_cache=use_cache,
    force_refresh=force_refresh,
)

spy_raw = data.download_data(
    [benchmark_ticker],
    start_date,
    end_date,
    use_cache=use_cache,
    force_refresh=force_refresh,
)

print(f'Universe rows: {len(df_raw):,}')
print(f'Benchmark rows: {len(spy_raw):,}')
df_raw.head()

## 4. Feature Engineering And Labeling

Features are computed per ticker using only information available by the close of day `t`. The target is next-session intraday direction. Final rows with unknown `t+1` outcomes remain `NaN` and are excluded from training and calibration.

In [ ]:
df = features.create_features(df_raw)
df = features.create_target(df)
feature_cols = features.get_feature_columns()

print('ML feature columns:')
for col in feature_cols:
    print(f'- {col}')

label_summary = df['target'].value_counts(dropna=False).rename('count').to_frame()
label_summary

In [ ]:
df.reset_index().tail(10)[[
    'ticker', 'Date', 'Open', 'Close', 'return_5d', 'target'
]]

## 5. Walk-Forward Validation

Each fold retrains the model using an expanding history and predicts only the next out-of-sample window. Capital carries forward across folds, so the stitched equity curve reflects a deployable sequence rather than independent test slices.

In [ ]:
wf_kwargs = dict(
    feature_cols=feature_cols,
    initial_capital=initial_capital,
    max_positions=max_positions,
    max_weight=max_weight,
    transaction_cost=transaction_cost,
    return_threshold=return_threshold,
    min_train_years=min_train_years,
    test_window_months=test_window_months,
    risk_free_rate=risk_free_rate,
)

raw_equity, raw_trades, raw_metrics, raw_folds, _ = walkforward.run_walk_forward(
    df,
    ml_prob_threshold=None,
    **wf_kwargs,
)

ml_equity, ml_trades, ml_metrics, ml_folds, oos_preds = walkforward.run_walk_forward(
    df,
    ml_prob_threshold=ml_prob_threshold,
    **wf_kwargs,
)

print(f'Raw trades: {len(raw_trades):,}')
print(f'ML trades: {len(ml_trades):,}')
print(f'ML OOS prediction samples: {0 if oos_preds is None else len(oos_preds["actual"]):,}')

## 6. Fold-Level Results

A strategy that only works in one market regime is less credible than one with repeatable fold-level behavior. Review consistency before reading the aggregate performance numbers.

In [ ]:
pd.DataFrame(ml_folds)

## 7. Strategy Comparison

All-period Sharpe includes flat cash days. Active-period Sharpe is shown separately to isolate performance on days where the strategy actually deployed capital.

In [ ]:
metrics.compare_strategies(raw_metrics, ml_metrics)

## 8. Equity Curves Versus SPY

SPY is normalized to the same out-of-sample start date as the walk-forward backtest.

In [ ]:
backtest_start = (
    pd.Timestamp(start_date) + pd.DateOffset(years=min_train_years)
).strftime('%Y-%m-%d')

spy_close = spy_raw.reset_index(level='ticker', drop=True)['Close']
spy_equity = metrics.compute_spy_equity(spy_close, initial_capital, backtest_start)

plots.plot_equity_curves(
    raw_equity,
    ml_equity,
    initial_capital=initial_capital,
    spy_equity=spy_equity,
)

## 9. Calendar-Year Returns

Yearly returns help identify whether the aggregate result is broad-based or dominated by a small number of periods.

In [ ]:
df_backtest_period = df[
    df.index.get_level_values('Date') >= pd.Timestamp(backtest_start)
]

raw_complete = metrics.get_complete_equity(raw_equity, df_backtest_period, initial_capital)
ml_complete = metrics.get_complete_equity(ml_equity, df_backtest_period, initial_capital)

raw_yearly = metrics.get_yearly_returns(raw_complete)
ml_yearly = metrics.get_yearly_returns(ml_complete)
spy_yearly = metrics.get_yearly_returns(spy_equity)

plots.plot_yearly_returns(raw_yearly, ml_yearly, spy_yearly)

In [ ]:
yearly_table = raw_yearly[['Year', 'Return']].rename(columns={'Return': 'Raw'})
yearly_table = yearly_table.merge(
    ml_yearly[['Year', 'Return']].rename(columns={'Return': 'ML'}),
    on='Year',
    how='outer',
)
yearly_table = yearly_table.merge(
    spy_yearly[['Year', 'Return']].rename(columns={'Return': 'SPY'}),
    on='Year',
    how='outer',
)
yearly_table

## 10. Risk And Trade Diagnostics

Drawdown and trade P&L distribution show the path and payoff shape behind the headline return.

In [ ]:
plots.plot_drawdowns(ml_equity, 'ML Strategy')

In [ ]:
plots.plot_trade_distribution(ml_trades, 'ML Strategy')

## 11. Probability Calibration

The ML filter uses a fixed threshold of `0.50`. This is defensible only if the model's probabilities are reasonably calibrated. The calibration check below uses out-of-sample walk-forward predictions, not training data.

In [ ]:
cal_data = None
if oos_preds is not None:
    cal_data = model.check_calibration_oos(oos_preds['proba'], oos_preds['actual'])

if cal_data is None:
    print('No out-of-sample predictions available for calibration.')
else:
    print(f"ECE: {cal_data['ece']:.4f}")
    print(f"Brier score: {cal_data['brier_score']:.4f}")
    print(f"Base rate: {cal_data['base_rate']:.2%}")
    plots.plot_calibration_curve(cal_data)

## 12. Feature Importance

The coefficient table is from a reference model trained on the history before the first out-of-sample date. Positive coefficients increase the predicted probability that `Close[t+1] > Open[t+1]`.

In [ ]:
ref_model, ref_scaler, ref_train_data = model.train_model(
    df,
    feature_cols,
    backtest_start,
)

model.get_feature_importance(ref_model, feature_cols)

## 13. Recent Trades

The trade log records signal date, selected ticker, position weight, gross intraday return, cost-adjusted return, P&L, and before/after capital.

In [ ]:
if len(ml_trades) == 0:
    print('No ML trades generated for this parameter set.')
else:
    ml_trades.tail(20)

## 14. Research Notes

Interpretation checklist:

- Does the ML filter improve risk-adjusted performance versus the raw signal?
- Is performance distributed across folds or concentrated in one period?
- Is drawdown acceptable relative to the simple SPY benchmark?
- Does calibration support the fixed `0.50` probability threshold?
- Are there enough trades to make the result statistically meaningful?

Known limitations:

- The universe is survivorship-biased.
- Daily bars do not model opening auction slippage or market impact.
- The cost model is static.
- The universe is small and concentrated in mega-cap technology stocks.